In [ ]:
import os, random, warnings, csv as _csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
def seed_everything(seed=7):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(7)
print('Seed = 7')

In [ ]:
# CONFIG

# Original training CSVs (3 known rock classes, original data only)
ROCK_CSVS_DIR = '/home/puneeth/Desktop/rock_csvs'

# New rock profile CSVs 
NEW_ROCK_CSVS_DIR = '/home/puneeth/Desktop/profiles_csv'

# Saved autoencoder model (from ood_autoencoder_1d.ipynb)
AUTOENCODER_PATH = 'results_ood_autoencoder_1d/autoencoder_1d_multisource.pth'

RESULTS_DIR = 'results_1d_cnn_inference'
DIR_TRAIN   = os.path.join(RESULTS_DIR, 'training')
DIR_EVAL    = os.path.join(RESULTS_DIR, 'evaluation')
DIR_INF     = os.path.join(RESULTS_DIR, 'inference')
for d in [RESULTS_DIR, DIR_TRAIN, DIR_EVAL, DIR_INF]:
    os.makedirs(d, exist_ok=True)

MODEL_183 = os.path.join(RESULTS_DIR, '1d_cnn_1-83Hz.pth')
MODEL_510 = os.path.join(RESULTS_DIR, '1d_cnn_5-10Hz.pth')

SPECTRUM_LEN = 1060
EPOCHS       = 50
LR           = 1e-3
BATCH_SIZE   = 64
TEST_SPLIT   = 0.20

# OOD threshold from autoencoder notebook
AE_OOD_THRESHOLD = 0.01190767   # <- update this if you reran the autoencoder
AE_LATENT_DIM    = 32           # must match what was used to train the autoencoder

CLASS_NAMES  = ['S10Granite', 'Holstein_Sandstone', 'Leitendorf_Limestone']
SHORT_NAMES  = ['Granite', 'Sandstone', 'Limestone']
CLASS_COLORS = ['#4e79a7', '#f28e2b', '#59a14f']
OOD_COLOR    = '#e15759'
VALID_EXT    = ('.csv',)

# Expected class for each new rock folder CSV
EXPECTED_CLASS = {
    'Dunite-Ecologite_2Rocks_1-83Hz'               : 'Unknown',
    'Gneis_1-83Hz'                                  : 'S10Granite',
    'Granite_3SamplesPhilipp_1-83Hz_1'              : 'S10Granite',
    'Granite_3SamplesPhilipp_1-83Hz_2'              : 'S10Granite',
    'Limestone_CalcsilicaContaminated_U9_U3_1-83Hz' : 'Leitendorf_Limestone',
    'Limestone_CalcsilicaContaminated_U9_U3_5-10Hz' : 'Leitendorf_Limestone',
    'Limestone_Rax_1-83Hz_1'                        : 'Leitendorf_Limestone',
    'Limestone_Rax_1-83Hz_2'                        : 'Leitendorf_Limestone',
    'SandstoneNew_1-83Hz'                           : 'Holstein_Sandstone',
}

_saved_files = []
def save_fig(fig, folder, filename, description, dpi=150):
    path = os.path.join(folder, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    _saved_files.append((path, description))
    print(f'[SAVED] {path}')

print('Config ready.')
print(f'  ROCK_CSVS_DIR     = {ROCK_CSVS_DIR}')
print(f'  NEW_ROCK_CSVS_DIR = {NEW_ROCK_CSVS_DIR}')
print(f'  AE_OOD_THRESHOLD  = {AE_OOD_THRESHOLD}')

In [ ]:
# CSV LOADER — same logic as the original 1D CNN notebooks
# Each CSV has 6 rows per spectrum block:
#   1. index row (text)    <- skipped (non-numeric)
#   2. wave row  (text)    <- skipped (non-numeric)
#   3. intensity           <- LOADED
#   4. intensity (dup)     <- LOADED
#   5. zeros               <- skipped (all-zero filter)
#   6. baseline 3.0        <- skipped (max >= 3.0 filter)

def load_csv_spectra(csv_path, n_points=SPECTRUM_LEN):
    spectra = []
    with open(csv_path, 'r') as f:
        reader = _csv.reader(f)
        for row in reader:
            if len(row) < n_points: continue
            try:
                vals = [float(v.replace(',','.').replace('+',''))
                        for v in row[:n_points]]
                if all(v == 0.0 for v in vals): continue
                if max(vals) >= 3.0:            continue
                spectra.append(vals)
            except ValueError:
                continue
    return np.array(spectra, dtype=np.float32) if spectra else None


# Auto-discover training CSV files
def find_csv(csvs_dir, class_name, speed_tag):
    key = class_name.lower().replace('_','')
    spd = speed_tag.replace('.','-')
    matches = [
        f for f in os.listdir(csvs_dir)
        if f.endswith('.csv')
        and spd in f.replace('.','-')
        and any(part in f.lower().replace('_','')
                for part in [key, class_name.split('_')[0].lower()])
    ]
    if not matches:
        raise FileNotFoundError(
            f'No CSV for class={class_name} speed={speed_tag}\n'
            f'Available: {sorted(os.listdir(csvs_dir))}')
    return os.path.join(csvs_dir, sorted(matches)[0])


# Load original training data
print('=== Loading original training CSVs (1.83 Hz) ===')
spectra_train_183 = {}
for cls in CLASS_NAMES:
    path = find_csv(ROCK_CSVS_DIR, cls, '1-83')
    arr  = load_csv_spectra(path)
    spectra_train_183[cls] = arr
    print(f'  {cls}: {arr.shape[0]} spectra  <- {os.path.basename(path)}')

print('\n=== Loading original training CSVs (5.10 Hz) ===')
spectra_train_510 = {}
for cls in CLASS_NAMES:
    path = find_csv(ROCK_CSVS_DIR, cls, '5-10')
    arr  = load_csv_spectra(path)
    spectra_train_510[cls] = arr
    print(f'  {cls}: {arr.shape[0]} spectra  <- {os.path.basename(path)}')

In [ ]:
# NORMALIZE + BUILD TRAINING ARRAYS
def normalize_spectra(X):
    mn = X.min(axis=1, keepdims=True)
    mx = X.max(axis=1, keepdims=True)
    d  = mx - mn; d[d == 0] = 1.0
    return (X - mn) / d


def build_xy(spectra_dict):
    X, y = [], []
    for ci, cls in enumerate(CLASS_NAMES):
        arr = normalize_spectra(spectra_dict[cls].copy())
        X.append(arr)
        y.extend([ci] * len(arr))
    return np.vstack(X), np.array(y)


X_183, y_183 = build_xy(spectra_train_183)
X_510, y_510 = build_xy(spectra_train_510)

print(f'1.83 Hz: {X_183.shape[0]} spectra  '
      f'{[(c,int(np.sum(y_183==i))) for i,c in enumerate(CLASS_NAMES)]}')
print(f'5.10 Hz: {X_510.shape[0]} spectra  '
      f'{[(c,int(np.sum(y_510==i))) for i,c in enumerate(CLASS_NAMES)]}')

In [ ]:
# PLOT SAMPLE SPECTRA
# What it shows: one representative normalised spectrum per class for each speed.
# Confirms the spectral shapes are distinct and physically interpretable.

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    'Sample Normalised Training Spectra\n'
    'Top = 1.83 Hz  |  Bottom = 5.10 Hz  |  '
    'Peak positions are mineralogy-invariant across quarries',
    fontsize=11, fontweight='bold')
for row, (speed, spectra_dict) in enumerate([
    ('1.83 Hz', spectra_train_183),
    ('5.10 Hz', spectra_train_510)
]):
    for ci, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
        ax  = axes[row][ci]
        arr = normalize_spectra(spectra_dict[cls].copy())
        s   = arr[len(arr)//2]
        ax.plot(s, color=color, lw=0.8)
        ax.fill_between(range(len(s)), s, alpha=0.2, color=color)
        ax.set_title(f'{SHORT_NAMES[ci]} [{speed}]', fontsize=9,
                     fontweight='bold')
        ax.set_xlabel('Wavenumber index')
        ax.set_ylabel('Normalised intensity')
        ax.set_ylim(-0.05, 1.1); ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, DIR_TRAIN,
    'TR-00_sample_training_spectra.png',
    'Sample normalised training spectra per class per belt speed.')
plt.show()

In [ ]:
# 1D CNN MODEL
# Architecture from 1d_cnn_per_speed_csv.ipynb:
# 3 x (Conv1d -> BatchNorm -> ReLU -> MaxPool -> Dropout)
# -> LayerNorm -> Linear(256->64) -> ReLU -> Dropout -> Linear(64->3)

class CNN1D(nn.Module):
    def __init__(self, input_len=1060, n_classes=3):
        super().__init__()
        self.conv_block = nn.Sequential(
            # Block 1
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
            # Block 2
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
            # Block 3 
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
        )
        # Compute flattened size after 3 MaxPool(2) on input_len
        conv_out = input_len // 8  # 3 x MaxPool(2) = divide by 8
        self.classifier = nn.Sequential(
            nn.LayerNorm(64 * conv_out),
            nn.Linear(64 * conv_out, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        x = x.flatten(1)
        return self.classifier(x)


_m = CNN1D(SPECTRUM_LEN, len(CLASS_NAMES))
n  = sum(p.numel() for p in _m.parameters())
print(f'1D CNN | Parameters: {n:,}')
del _m

In [ ]:
# TRAINING FUNCTION
def train_1d_cnn(X, y, model_save_path, speed_tag):
    if os.path.exists(model_save_path):
        print(f'[SKIP] {os.path.basename(model_save_path)} exists. '
              f'Delete to retrain.')
        return None

    seed_everything(7)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SPLIT, stratify=y, random_state=7)

    print(f'\nTraining 1D CNN [{speed_tag}]  '
          f'n_train={len(X_tr)}  n_val={len(X_te)}')

    tr_t = torch.tensor(X_tr, dtype=torch.float32)
    te_t = torch.tensor(X_te, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long)
    y_te_t = torch.tensor(y_te, dtype=torch.long)

    nw  = min(4, os.cpu_count() or 1)
    pin = (device.type == 'cuda')
    tr_ldr = DataLoader(
        TensorDataset(tr_t, y_tr_t), BATCH_SIZE,
        shuffle=True, num_workers=nw, pin_memory=pin,
        persistent_workers=(nw > 0))
    te_ldr = DataLoader(
        TensorDataset(te_t, y_te_t), BATCH_SIZE,
        shuffle=False, num_workers=nw, pin_memory=pin,
        persistent_workers=(nw > 0))

    model     = CNN1D(SPECTRUM_LEN, len(CLASS_NAMES)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()

    tr_accs, te_accs = [], []
    best_acc = -1.0

    for epoch in range(EPOCHS):
        model.train()
        ep_acc = []
        for Xb, yb in tr_ldr:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(Xb); loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            ep_acc.append((out.argmax(1)==yb).float().mean().item())
        tr_accs.append(float(np.mean(ep_acc)))

        model.eval()
        va = []
        with torch.no_grad():
            for Xb, yb in te_ldr:
                Xb, yb = Xb.to(device), yb.to(device)
                va.append((model(Xb).argmax(1)==yb).float().mean().item())
        val_acc = float(np.mean(va))
        te_accs.append(val_acc)
        scheduler.step(val_acc)

        saved = ''
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), model_save_path)
            saved = ' [SAVED]'

        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f'  Ep {epoch+1:>3}/{EPOCHS}  '
                  f'train={tr_accs[-1]*100:.2f}%  '
                  f'val={val_acc*100:.2f}%{saved}')

    print(f'  Best val accuracy: {best_acc*100:.2f}%  -> {model_save_path}')
    del model, optimizer, criterion
    torch.cuda.empty_cache()
    return {'tr_accs': tr_accs, 'te_accs': te_accs, 'best_acc': best_acc}

print('train_1d_cnn() defined.')

In [ ]:
hist_183 = train_1d_cnn(X_183, y_183, MODEL_183, '1.83 Hz')

In [ ]:
hist_510 = train_1d_cnn(X_510, y_510, MODEL_510, '5.10 Hz')

In [ ]:
# PLOT TRAINING HISTORY
valid = {k: v for k,v in [('1.83 Hz',hist_183),('5.10 Hz',hist_510)]
         if v is not None}
if valid:
    fig, axes = plt.subplots(1, len(valid), figsize=(8*len(valid), 4))
    if len(valid) == 1: axes = [axes]
    fig.suptitle('TR-01 | 1D CNN Training History\n'
                 'Trained on original 3 rock classes only',
                 fontsize=11, fontweight='bold')
    for ax, (tag, h) in zip(axes, valid.items()):
        ep = range(1, len(h['tr_accs'])+1)
        ax.plot(ep, np.array(h['tr_accs'])*100, 'b-', lw=1.5, label='Train')
        ax.plot(ep, np.array(h['te_accs'])*100,  'r-', lw=1.5, label='Val')
        ax.axhline(h['best_acc']*100, color='green', ls='--', lw=1.2,
                   label=f'Best: {h["best_acc"]*100:.2f}%')
        ax.set_title(tag); ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 105)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_fig(fig, DIR_TRAIN,
        'TR-01_1dcnn_training_history.png',
        '1D CNN training and validation accuracy curves.')
    plt.show()
else:
    print('Models loaded from disk.')

In [ ]:
# LOAD SAVED MODELS
def load_cnn(path):
    model = CNN1D(SPECTRUM_LEN, len(CLASS_NAMES)).to(device)
    model.load_state_dict(
        torch.load(path, map_location=device, weights_only=True))
    model.eval()
    return model

cnn_183 = load_cnn(MODEL_183)
cnn_510 = load_cnn(MODEL_510)
print(f'Loaded 1.83 Hz <- {MODEL_183}')
print(f'Loaded 5.10 Hz <- {MODEL_510}')

In [ ]:
# EVALUATE ON KNOWN ROCKS
def evaluate(model, X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SPLIT, stratify=y, random_state=7)
    t   = torch.tensor(X_te, dtype=torch.float32)
    ldr = DataLoader(TensorDataset(t), 256, shuffle=False)
    preds = []
    with torch.no_grad():
        for (Xb,) in ldr:
            preds.extend(model(Xb.to(device)).argmax(1).cpu().tolist())
    return np.array(preds), y_te

print('=== Evaluation on known rocks (validation split) ===')
rpts = {}
for tag, model, X, y in [
    ('1-83Hz', cnn_183, X_183, y_183),
    ('5-10Hz', cnn_510, X_510, y_510)
]:
    preds, labels = evaluate(model, X, y)
    acc = np.mean(preds == labels) * 100
    rpt = classification_report(labels, preds, target_names=CLASS_NAMES,
                                 output_dict=True, zero_division=0)
    rpts[tag] = rpt
    print(f'\n{tag}  acc={acc:.2f}%')
    print(classification_report(labels, preds,
                                  target_names=CLASS_NAMES, digits=3))

In [ ]:
# EV-01  Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'EV-01 | 1D CNN Confusion Matrices on Known Rocks (Validation Split)\n'
    'Trained on original 3 classes only — no new rocks in training data',
    fontsize=11, fontweight='bold')
for ax, (tag, model, X, y) in zip(axes, [
    ('1.83 Hz', cnn_183, X_183, y_183),
    ('5.10 Hz', cnn_510, X_510, y_510)
]):
    preds, labels = evaluate(model, X, y)
    cm  = confusion_matrix(labels, preds)
    acc = np.mean(preds == labels) * 100
    sns.heatmap(cm, ax=ax, annot=True, fmt='d', cmap='Blues',
                xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES)
    ax.set_title(f'{tag}  acc={acc:.2f}%', fontsize=11)
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
save_fig(fig, DIR_EVAL,
    'EV-01_confusion_matrices__known_rocks.png',
    '1D CNN confusion matrices on known rock validation split.')
plt.show()

In [ ]:
# LOAD AUTOENCODER
class SpectralAutoencoder(nn.Module):
    def __init__(self, input_dim=1060, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LeakyReLU(0.1),
            nn.Linear(512, 128),       nn.LeakyReLU(0.1),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.LeakyReLU(0.1),
            nn.Linear(128, 512),        nn.LeakyReLU(0.1),
            nn.Linear(512, input_dim),  nn.Sigmoid(),
        )
    def forward(self, x): return self.decoder(self.encoder(x))
    def mse(self, x):
        with torch.no_grad():
            return ((self.forward(x) - x)**2).mean(dim=1)


if os.path.exists(AUTOENCODER_PATH):
    ae = SpectralAutoencoder(SPECTRUM_LEN, AE_LATENT_DIM).to(device)
    ae.load_state_dict(
        torch.load(AUTOENCODER_PATH, map_location=device, weights_only=True))
    ae.eval()
    print(f'Autoencoder loaded <- {AUTOENCODER_PATH}')
    print(f'OOD threshold: {AE_OOD_THRESHOLD}')
    USE_AE = True
else:
    print(f'[WARNING] Autoencoder not found at {AUTOENCODER_PATH}')
    print('OOD detection will be disabled — all spectra will be classified.')
    print('Run ood_autoencoder_1d.ipynb first to generate the autoencoder.')
    USE_AE = False

In [ ]:
# LOAD NEW ROCK SPECTRA
new_rock_spectra = {}
new_rock_failed  = []

print('Loading new rock CSVs...')
csv_files = sorted(os.listdir(NEW_ROCK_CSVS_DIR))
print(f'Found {len(csv_files)} CSV files in {NEW_ROCK_CSVS_DIR}')

for fn, exp in EXPECTED_CLASS.items():
    csv_path = os.path.join(NEW_ROCK_CSVS_DIR, fn + '.csv')
    if not os.path.exists(csv_path):
        print(f'  [SKIP] {fn} — CSV not found')
        new_rock_failed.append(fn)
        continue
    arr = load_csv_spectra(csv_path)
    if arr is None or len(arr) == 0:
        print(f'  [SKIP] {fn} — 0 spectra loaded')
        new_rock_failed.append(fn)
        continue
    arr_norm = arr.copy()
    mn = arr_norm.min(axis=1, keepdims=True)
    mx = arr_norm.max(axis=1, keepdims=True)
    d  = mx - mn; d[d==0] = 1.0
    arr_norm = (arr_norm - mn) / d
    new_rock_spectra[fn] = arr_norm
    print(f'  {fn}: {arr_norm.shape[0]} spectra  (expected: {exp})')

print(f'\nLoaded: {len(new_rock_spectra)} folders')

In [ ]:
# RUN INFERENCE WITH OOD PRE-FILTER
# For each new rock folder:
#   1. Compute autoencoder MSE per spectrum
#   2. If MSE > AE_OOD_THRESHOLD -> Unknown
#   3. Else -> feed to 1D CNN -> predict class + confidence

def infer_with_ood(cnn_model, spectra_norm, folder_name):
    use_510 = '5-10Hz' in folder_name
    cnn     = cnn_510 if use_510 else cnn_183
    t       = torch.tensor(spectra_norm, dtype=torch.float32)
    results = []

    # Step 1: autoencoder OOD check
    if USE_AE:
        ldr = DataLoader(TensorDataset(t), 256, shuffle=False)
        ae_errors = []
        with torch.no_grad():
            for (Xb,) in ldr:
                ae_errors.extend(ae.mse(Xb.to(device)).cpu().tolist())
        ae_errors = np.array(ae_errors)
    else:
        ae_errors = np.zeros(len(spectra_norm))

    # Step 2: 1D CNN inference on non-OOD spectra
    ldr = DataLoader(TensorDataset(t), 256, shuffle=False)
    cnn_probs = []
    cnn.eval()
    with torch.no_grad():
        for (Xb,) in ldr:
            probs = torch.softmax(cnn(Xb.to(device)), dim=1).cpu().numpy()
            cnn_probs.extend(probs.tolist())
    cnn_probs = np.array(cnn_probs)

    for i in range(len(spectra_norm)):
        ae_err  = float(ae_errors[i])
        is_ood  = bool(ae_err > AE_OOD_THRESHOLD)
        prob    = cnn_probs[i]
        pred_idx = int(np.argmax(prob))
        conf    = float(np.max(prob))
        results.append({
            'pred_class'  : 'Unknown' if is_ood else CLASS_NAMES[pred_idx],
            'confidence'  : conf,
            'ae_error'    : ae_err,
            'is_ood'      : is_ood,
            'probs'       : prob.tolist(),
        })
    return results


folder_results = {}
print('Running inference on new rock folders...')
for fn, spectra in new_rock_spectra.items():
    res       = infer_with_ood(cnn_183, spectra, fn)
    n_ood     = sum(1 for r in res if r['is_ood'])
    top_cls   = max(set(r['pred_class'] for r in res),
                    key=lambda c: sum(1 for r in res if r['pred_class']==c))
    mean_conf = np.mean([r['confidence'] for r in res])
    exp       = EXPECTED_CLASS.get(fn,'?')
    folder_results[fn] = res
    print(f'  {fn}')
    print(f'    top={top_cls.split("_")[0]}  '
          f'conf={mean_conf*100:.1f}%  '
          f'OOD={n_ood}/{len(res)}  '
          f'expected={exp.split("_")[0]}')

In [ ]:
# INF-01  Prediction distribution per folder
display_classes = CLASS_NAMES + ['Unknown']
display_colors  = CLASS_COLORS + [OOD_COLOR]
display_short   = SHORT_NAMES + ['Unknown']

folder_names = list(folder_results.keys())
n_folders    = len(folder_names)
short_labels = [
    fn.replace('Limestone_','Lst_')
      .replace('Granite_3SamplesPhilipp_','Gran_Phil_')
      .replace('SandstoneNew','SstNew')
      .replace('Dunite-Ecologite_2Rocks_','Dun-Eco_')
      .replace('CalcsilicaContaminated_U9_U3_','CalcSil_')
    for fn in folder_names
]

fig, ax = plt.subplots(figsize=(max(14, n_folders*2), 8))
fig.suptitle(
    'INF-01 | 1D CNN Prediction Distribution per New Rock Folder\n'
    'Trained on original 3 classes only — no new rocks in training\n'
    'Red = flagged as Unknown by autoencoder OOD pre-filter  |  '
    'Italic below = expected class',
    fontsize=11, fontweight='bold')

x = np.arange(n_folders); bottom = np.zeros(n_folders)
for cls, color, short in zip(display_classes, display_colors, display_short):
    fracs = [
        sum(1 for r in folder_results[fn] if r['pred_class']==cls)
        / len(folder_results[fn]) * 100
        for fn in folder_names
    ]
    ax.bar(x, fracs, bottom=bottom, color=color, label=short, width=0.65)
    for xi, (frac, bot) in enumerate(zip(fracs, bottom)):
        if frac > 7:
            ax.text(xi, bot+frac/2, f'{frac:.0f}%', ha='center',
                    va='center', fontsize=7.5, color='white', fontweight='bold')
    bottom += np.array(fracs)

for xi, fn in enumerate(folder_names):
    res   = folder_results[fn]
    conf  = np.mean([r['confidence'] for r in res])*100
    n_ood = sum(1 for r in res if r['is_ood'])
    ax.text(xi, 102, f'{conf:.0f}%\nconf', ha='center', va='bottom',
            fontsize=7, color='#333')
    if n_ood > 0:
        ax.text(xi, 111, f'\u26a0 {n_ood}', ha='center', va='bottom',
                fontsize=7, color='red')
    exp = EXPECTED_CLASS.get(fn,'?').split('_')[0]
    ax.text(xi, -9, f'exp:\n{exp}', ha='center', va='top',
            fontsize=6.5, color='navy', style='italic')

ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('% of spectra'); ax.set_ylim(-15, 122)
ax.set_yticks(range(0,101,20))
ax.legend(title='Predicted', loc='upper right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.axhline(100, color='black', lw=0.5); ax.axhline(0, color='black', lw=0.3)
plt.tight_layout()
save_fig(fig, DIR_INF,
    'INF-01_1dcnn_prediction_distribution__new_rocks.png',
    '1D CNN prediction distribution per new rock folder with OOD pre-filter.')
plt.show()

In [ ]:
# INF-02  Confidence heatmap
mean_confs = [np.mean([r['confidence'] for r in folder_results[fn]])*100
              for fn in folder_names]
pct_ood    = [sum(1 for r in folder_results[fn] if r['is_ood'])
              /len(folder_results[fn])*100
              for fn in folder_names]
mat = np.array([mean_confs, pct_ood]).T

fig, ax = plt.subplots(figsize=(5, max(5, n_folders*0.65)))
fig.suptitle(
    'INF-02 | 1D CNN Confidence & OOD Flagging\n'
    'Left = mean CNN confidence  |  Right = % flagged Unknown by autoencoder',
    fontsize=10, fontweight='bold')
sns.heatmap(mat, ax=ax,
            xticklabels=['Mean\nConfidence (%)', 'OOD Flagged\n(%)'],
            yticklabels=short_labels,
            annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100,
            linewidths=0.5, linecolor='white')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
save_fig(fig, DIR_INF,
    'INF-02_1dcnn_confidence_heatmap__new_rocks.png',
    '1D CNN mean confidence and % OOD-flagged per new rock folder.')
plt.show()

In [ ]:
# FINAL SUMMARY TABLE
def get_verdict(fn):
    res     = folder_results[fn]
    top_cls = max(set(r['pred_class'] for r in res),
                  key=lambda c: sum(1 for r in res if r['pred_class']==c))
    exp_cls = EXPECTED_CLASS.get(fn, '?')
    if exp_cls=='Unknown' and top_cls=='Unknown': return '\u2705 Correctly Unknown'
    if exp_cls=='Unknown' and top_cls!='Unknown': return '\u274c Missed OOD'
    if exp_cls!='Unknown' and top_cls=='Unknown': return '\u26a0 False OOD'
    if top_cls==exp_cls:                           return '\u2705 Correct'
    return '\u274c Incorrect'

print(f'\n{"="*100}')
print('1D CNN + AUTOENCODER OOD — FINAL RESULTS')
print(f'Trained on original 3 classes only  |  '
      f'OOD threshold: {AE_OOD_THRESHOLD}')
print(f'{"="*100}')
print(f'{"Folder":<48} {"Expected":<12} {"Predicted":<12} '
      f'{"OOD%":>6} {"Conf%":>7}  Result')
print('-'*100)

for fn in folder_names:
    res     = folder_results[fn]
    top_cls = max(set(r['pred_class'] for r in res),
                  key=lambda c: sum(1 for r in res if r['pred_class']==c))
    exp_cls = EXPECTED_CLASS.get(fn,'?')
    n_ood   = sum(1 for r in res if r['is_ood'])
    conf    = np.mean([r['confidence'] for r in res])*100
    verdict = get_verdict(fn)
    lbl     = (fn.replace('Limestone_','Lst_')
                 .replace('Granite_3SamplesPhilipp_','Gran_Phil_')
                 .replace('SandstoneNew','SstNew')
                 .replace('Dunite-Ecologite_2Rocks_','Dun-Eco_')
                 .replace('CalcsilicaContaminated_U9_U3_','CalcSil_'))
    print(f'{lbl:<48} {exp_cls.split("_")[0]:<12} '
          f'{top_cls.split("_")[0]:<12} '
          f'{n_ood/len(res)*100:>5.1f}% {conf:>6.1f}%  {verdict}')

print('='*100)

In [ ]:
# SAVE CSV
csv_path = os.path.join(RESULTS_DIR, 'predictions_1dcnn_new_rocks.csv')
with open(csv_path,'w',newline='') as f:
    w = _csv.writer(f)
    w.writerow(['folder','expected_class','top_predicted_class',
                'mean_confidence_pct','n_spectra','n_ood','pct_ood',
                'pct_Granite','pct_Sandstone','pct_Limestone','pct_Unknown',
                'verdict'])
    for fn in folder_names:
        res    = folder_results[fn]
        top    = max(set(r['pred_class'] for r in res),
                     key=lambda c: sum(1 for r in res if r['pred_class']==c))
        n_ood  = sum(1 for r in res if r['is_ood'])
        conf   = np.mean([r['confidence'] for r in res])*100
        cts    = {c: sum(1 for r in res if r['pred_class']==c)
                  for c in CLASS_NAMES+['Unknown']}
        w.writerow([
            fn, EXPECTED_CLASS.get(fn,'?'), top,
            round(conf,2), len(res), n_ood,
            round(n_ood/len(res)*100,2),
            round(cts['S10Granite']/len(res)*100,2),
            round(cts['Holstein_Sandstone']/len(res)*100,2),
            round(cts['Leitendorf_Limestone']/len(res)*100,2),
            round(cts['Unknown']/len(res)*100,2),
            get_verdict(fn)
        ])
_saved_files.append((csv_path, 'Per-folder 1D CNN inference results'))
print(f'[SAVED] {csv_path}')
print(f'Total files: {len(_saved_files)}')